# 原时间有限支持 KL：固定 rho 与 adaptive rho

所有模型直接在原时间 `[0, upper_bound]` 上划分叶区间。
`upper_bound = 1.1 * max_observation`，其中最大值仅取当前拟合子集的有限观测 L/R 端点。
CV 每折独立估计上界；最终只从前80%拟合池估计上界，不读取验证/测试或潜在事件时间。
不使用非线性归一化。保留 KL 的 `1/L`、`2/depth`、`a/sqrt(n_fit)`，目标叶宽为 `upper_bound/L`。

仍使用35 setting、两组独立64-trial四折 NLL 搜索，另含匹配架构固定 rho 对照。
所有选参冻结后，seed0–9按固定 CV refit epoch 拟合，测试不参与选参或早停。
SSE 仍在256点 `[0,max_observation]` 原时间网格上逐对象求和后平均。
新输出目录为 `results/nll_cv_raw_time_comparison`。

修改前完整执行的无限支持 notebook 原样归档于
`archive/infinite_support_20261011/tune_nll_cv_rho.ipynb`；旧结果没有覆盖。
当前新 notebook 尚未执行完整 HPO。


In [ ]:
from pathlib import Path
import sys
import torch
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
while not (ROOT/'pycode'/'unified_nll').is_dir():
    if ROOT == ROOT.parent:
        raise RuntimeError('Run inside PolyaPFN')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0,str(ROOT))
from pycode.unified_nll.experiment import ExperimentConfig
from pycode.unified_nll.train import TrainingConfig
from pycode.unified_nll.tuning_model import Architecture
from pycode.unified_nll.cv_search import CVSearchConfig
from pycode.unified_nll.nll_cv_train import NLLCVOptimization
from pycode.unified_nll.raw_time.search import tune_nll_cv,evaluate_nll_cv,METHODS
torch.set_num_threads(4)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT = ROOT/'pycode'/'unified_nll'/'results'/'nll_cv_raw_time_comparison'
CONFIG,SEARCH = ExperimentConfig(),CVSearchConfig()
TRAINING,OPTIMIZATION,BASELINE = TrainingConfig(patience=50),NLLCVOptimization(),Architecture()
from pycode.unified_nll.data_compatibility import verify_generator
assert verify_generator()["passed"]
KWARGS = dict(config=CONFIG,search=SEARCH,training=TRAINING,optimization=OPTIMIZATION,
              baseline=BASELINE,device=DEVICE)
print('Device:',DEVICE,'Tuning seed:',CONFIG.data_seed,'Final DATA seeds:',CONFIG.seeds)
print('Methods:',METHODS)
print('CV fits:',35*(2*SEARCH.trials+1)*SEARCH.folds,'Final fits:',35*3*len(CONFIG.seeds))


## NLL CV through PH

Every candidate completes four folds; interrupted folds resume in place.

In [ ]:
selected,trials,folds = tune_nll_cv(OUTPUT,stop_after_settings=7,**KWARGS)
print('Settings completed:',selected.setting.nunique(),'Search candidates:',len(trials),'Fold fits:',len(folds))
display(selected.tail(21)[['setting','method','depth','shared_width','shared_layers',
    'private_width','private_layers','regularization_strength','refit_epochs','cv_nll']])


## NLL CV through lognormal

Every candidate completes four folds; interrupted folds resume in place.

In [ ]:
selected,trials,folds = tune_nll_cv(OUTPUT,stop_after_settings=14,**KWARGS)
print('Settings completed:',selected.setting.nunique(),'Search candidates:',len(trials),'Fold fits:',len(folds))
display(selected.tail(21)[['setting','method','depth','shared_width','shared_layers',
    'private_width','private_layers','regularization_strength','refit_epochs','cv_nll']])


## NLL CV through PO

Every candidate completes four folds; interrupted folds resume in place.

In [ ]:
selected,trials,folds = tune_nll_cv(OUTPUT,stop_after_settings=21,**KWARGS)
print('Settings completed:',selected.setting.nunique(),'Search candidates:',len(trials),'Fold fits:',len(folds))
display(selected.tail(21)[['setting','method','depth','shared_width','shared_layers',
    'private_width','private_layers','regularization_strength','refit_epochs','cv_nll']])


## NLL CV through multimodal separate

Every candidate completes four folds; interrupted folds resume in place.

In [ ]:
selected,trials,folds = tune_nll_cv(OUTPUT,stop_after_settings=28,**KWARGS)
print('Settings completed:',selected.setting.nunique(),'Search candidates:',len(trials),'Fold fits:',len(folds))
display(selected.tail(21)[['setting','method','depth','shared_width','shared_layers',
    'private_width','private_layers','regularization_strength','refit_epochs','cv_nll']])


## NLL CV through multimodal close

Every candidate completes four folds; interrupted folds resume in place.

In [ ]:
selected,trials,folds = tune_nll_cv(OUTPUT,stop_after_settings=35,**KWARGS)
print('Settings completed:',selected.setting.nunique(),'Search candidates:',len(trials),'Fold fits:',len(folds))
display(selected.tail(21)[['setting','method','depth','shared_width','shared_layers',
    'private_width','private_layers','regularization_strength','refit_epochs','cv_nll']])


## Frozen configurations → ten new train/test pairs

Parameters are selected once on seed123 and reused across ten independent
datasets. The reported SD includes data and initialization variation,
conditional on this tuning dataset. No test-based epoch selection.


In [ ]:
runs,summary = evaluate_nll_cv(OUTPUT,**KWARGS)
print('Final fits:',len(runs),'Data seeds:',sorted(runs.data_seed.unique()))
assert runs.n_fit.eq(4000).all() and runs.n_test.eq(1000).all()
display(summary)


In [ ]:
from pycode.unified_nll.raw_time.report import report_nll_cv,audit_nll_cv
overall,family,paired = report_nll_cv(OUTPUT)
display(overall)
display(family)
with pd.option_context('display.max_rows',100,'display.max_columns',12):
    display(paired)
print('Report:',OUTPUT/'grid_sse_report.md')
print(audit_nll_cv(OUTPUT))


In [ ]:
import matplotlib.pyplot as plt
from pycode.unified_nll.raw_time.report import plot_nll_cv
plot_nll_cv(OUTPUT)
plt.show()


## Interpretation

The two independently tuned winners can differ in architecture as well as
rho. The matched-architecture control keeps adaptive's network and depth,
with fixed rho and a separately CV-chosen refit duration. It does not match
training duration or trainable parameter count, so it is a practical
same-architecture comparison rather than a one-variable causal experiment.

64 trials per method is a finite search budget, not a global optimum claim.
NLL validation scores and test grid SSE measure different targets. One HPO
dataset per setting does not measure variability from repeating selection.
The ten final pairs match the preceding study's seeds: useful paired
comparisons, but not a new untouched benchmark after choosing NLL from
earlier test evidence. All model selection in this notebook uses CV NLL only.
